# Assignment 2 — Pandas: messy data handling

**Dataset:** Steam video games (scraped from SteamSpy), 26,688 rows × 10 columns.
- Source (TidyTuesday, 2019-07-30): https://github.com/rfordatascience/tidytuesday/tree/master/data/2019/2019-07-30
- Direct CSV link: https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2019/2019-07-30/video_games.csv

**Why it fits the requirements:** 26,688 rows; text columns `game`, `owners`, `developer`, `publisher`; date column `release_date`; numeric columns with real missing values (`price` ≈ 3,095 missing, `metascore` ≈ 23,838 missing).

In [1]:
import os, time
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 45)

URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2019/2019-07-30/video_games.csv"

DTYPES = {
    "number": "int64", "game": "object", "price": "float64", "owners": "object",
    "developer": "object", "publisher": "object",
    "average_playtime": "float64", "median_playtime": "float64", "metascore": "float64",
}

def read_source(src):
    return pd.read_csv(src, dtype=DTYPES, parse_dates=["release_date"], date_format="%b %d, %Y")

try:
    df = read_source(URL)
    print("Loaded from URL")
except Exception as e:
    print("URL failed, using local file:", e)
    df = read_source("video_games.csv")

print(df.shape)

Loaded from URL
(26688, 10)


## Q1. Load with explicit `dtype` / `parse_dates`, then reduce memory

In [2]:
print(df.dtypes)
print()
print("release_date dtype:", df["release_date"].dtype)
print("Sample values:", df["release_date"].head(3).tolist())

number                int64
game                 object
release_date         object
price               float64
owners               object
developer            object
publisher            object
average_playtime    float64
median_playtime     float64
metascore           float64
dtype: object

release_date dtype: object
Sample values: ['Nov 16, 2004', 'Nov 1, 2004', 'Mar 1, 2004']


**Observation:** `parse_dates` was requested, but `release_date` stays as text. About 42 values look like `Jun 2009` (no day), which do not fit the `%b %d, %Y` format, so pandas keeps the whole column as text instead of guessing. I deal with this properly in Q6 using `errors="coerce"`.

In [3]:
mem_before = df.memory_usage(deep=True).sum()
print(f"Memory BEFORE: {mem_before / 1024**2:.3f} MB")

df_opt = df.copy()

# 1) downcast numeric columns
num_cols = df_opt.select_dtypes(include="number").columns
for c in num_cols:
    kind = "integer" if pd.api.types.is_integer_dtype(df_opt[c]) else "float"
    df_opt[c] = pd.to_numeric(df_opt[c], downcast=kind)

# 2) low-cardinality text columns -> category (rule: unique values < 50% of rows)
text_cols = [c for c in df_opt.select_dtypes(include=["object", "string"]).columns if c != "release_date"]
ratio = pd.Series({c: df_opt[c].nunique() / len(df_opt) for c in text_cols}, name="unique_ratio")
print(ratio.round(4))
to_cat = ratio[ratio < 0.5].index.tolist()
for c in to_cat:
    df_opt[c] = df_opt[c].astype("category")
print("Converted to category:", to_cat)

mem_after = df_opt.memory_usage(deep=True).sum()
saving_pct = (mem_before - mem_after) / mem_before * 100
print(f"Memory AFTER : {mem_after / 1024**2:.3f} MB")
print(f"Saving       : {saving_pct:.1f} %")

print(df_opt.dtypes)
df = df_opt

Memory BEFORE: 9.555 MB
game         0.9971
owners       0.0005
developer    0.6277
publisher    0.5227
Name: unique_ratio, dtype: float64
Converted to category: ['owners']
Memory AFTER : 7.034 MB
Saving       : 26.4 %
number                 int16
game                  object
release_date          object
price                float32
owners              category
developer             object
publisher             object
average_playtime     float32
median_playtime      float32
metascore            float32
dtype: object


`developer` and `publisher` have too many different values (unique ratio above 0.5), so `category` would not save memory for them. Only `owners` (13 values) qualifies.

## Q2. Profiling DataFrame (one row per column)

In [4]:
def profile(d):
    top_val  = d.apply(lambda s: s.value_counts().index[0] if s.notna().any() else np.nan)
    top_freq = d.apply(lambda s: s.value_counts().iloc[0]  if s.notna().any() else 0)
    return pd.DataFrame({
        "dtype": d.dtypes.astype(str),
        "missing_count": d.isna().sum(),
        "missing_pct": (d.isna().mean() * 100).round(2),
        "unique_count": d.nunique(),
        "most_frequent": top_val,
        "frequency": top_freq,
    })

prof = profile(df)
prof

,dtype,missing_count,missing_pct,unique_count,most_frequent,frequency
number,int16,0,0.00,8846,8,15
game,object,3,0.01,26610,Dark Matter,3
release_date,object,0,0.00,2977,"Apr 5, 2016",63
price,float32,3095,11.60,199,9.99,3686
owners,category,0,0.00,13,"0 .. 20,000",17923
developer,object,152,0.57,16751,Choice of Games,92
publisher,object,120,0.45,13951,Big Fish Games,284
average_playtime,float32,9,0.03,386,0.0,25757
median_playtime,float32,12,0.04,296,0.0,25825
metascore,float32,23838,89.32,73,80.0,133


## Q3. Duplicates

In [5]:
exact = df.duplicated().sum()
print("Exact duplicate rows (all columns):", exact)

KEYS = ["game", "developer", "publisher"]
dup_mask = df.duplicated(subset=KEYS, keep=False)
print("Rows involved in a duplicate on keys", KEYS, ":", dup_mask.sum())
print("Extra copies (duplicated rows beyond the first):", df.duplicated(subset=KEYS).sum())
df[dup_mask].sort_values(KEYS)[["game", "developer", "publisher", "release_date", "price", "owners"]].head(10)

Exact duplicate rows (all columns): 0
Rows involved in a duplicate on keys ['game', 'developer', 'publisher'] : 82
Extra copies (duplicated rows beyond the first): 41


,game,developer,publisher,release_date,price,owners
305,18 Wheels of Steel: American Long Haul,SCS Software,"ValuSoft, Play Hard Games","Dec 3, 2007",9.99,"50,000 .. 100,000"
474,18 Wheels of Steel: American Long Haul,SCS Software,"ValuSoft, Play Hard Games","Jan 29, 2008",9.99,"50,000 .. 100,000"
69,Brothers in Arms: Earned in Blood,Gearbox Software,Ubisoft,"Oct 4, 2005",9.99,"100,000 .. 200,000"
434,Brothers in Arms: Earned in Blood,Gearbox Software,Ubisoft,"Oct 10, 2008",9.99,"100,000 .. 200,000"
68,Brothers in Arms: Road to Hill 30,Gearbox Software,Ubisoft,"Mar 15, 2005",9.99,"100,000 .. 200,000"
432,Brothers in Arms: Road to Hill 30,Gearbox Software,Ubisoft,"May 13, 2008",9.99,"100,000 .. 200,000"
578,Champions Online,Cryptic Studios,Perfect World Entertainment,"Sep 1, 2009",NaN,"1,000,000 .. 2,000,000"
1261,Champions Online,Cryptic Studios,Perfect World Entertainment,"Jun 14, 2011",NaN,"1,000,000 .. 2,000,000"
77,Crazy Machines,Fakt Software,Viva Media,"Oct 1, 2005",9.99,"50,000 .. 100,000"
462,Crazy Machines,Fakt Software,Viva Media,"Dec 12, 2008",9.99,"50,000 .. 100,000"


In [6]:
# Rule: keep the EARLIEST release_date for each key, drop later copies
sort_key = pd.to_datetime(df["release_date"], errors="coerce")
before = len(df)
df = (df.assign(_d=sort_key)
        .sort_values("_d", kind="stable")
        .drop_duplicates(subset=KEYS, keep="first")
        .sort_index()
        .drop(columns="_d")
        .reset_index(drop=True))
df["row_id"] = np.arange(len(df))          # unique key, needed later for merges (the 'number' column is NOT unique)
print(f"Rows before: {before}  after: {len(df)}  removed: {before - len(df)}")
print("Remaining duplicates on keys:", df.duplicated(subset=KEYS).sum())

Rows before: 26688  after: 26647  removed: 41
Remaining duplicates on keys: 0


**Rule kept:** for rows with the same `game` + `developer` + `publisher`, I keep the earliest `release_date`. These repeats are the same product listed again on a later date (identical price, owners and metascore), so the first listing is the original release and the later one is a re-listing. Keeping `game` alone would be wrong because different studios reuse titles such as *Alter Ego* and *Ashes*. There are 0 exact duplicates, so the key-based rule is the real cleaning step here.

## Q4. Clean one text column: `publisher`

In [7]:
s = df["publisher"].astype("object")
print("BEFORE: unique labels =", s.nunique())
print(s.value_counts().head(10))

BEFORE: unique labels = 13951
publisher
Big Fish Games         284
SEGA                   141
Strategy First         129
Ubisoft                116
Square Enix             95
Sekai Project           94
Choice of Games         92
Dagestan Technology     89
1C Entertainment        85
THQ Nordic              84
Name: count, dtype: int64


In [8]:
import re
SUFFIXES = r"\b(?:inc|ltd|llc|limited|co|corp|corporation|gmbh|oy|ab|bv|pty|company|sa|srl)\b"

# step 1: strip whitespace and collapse repeated spaces
s1 = s.str.strip().str.replace(r"\s+", " ", regex=True)

# step 2: build a normalised key (lower case, no punctuation, no company suffix)
key = (s1.str.lower()
         .str.replace(r"[^\w\s]", " ", regex=True)
         .str.replace(SUFFIXES, " ", regex=True)
         .str.replace(r"\s+", " ", regex=True)
         .str.strip())
key = key.mask(key == "", s1.str.lower())          # never leave an empty key

# step 3: every key maps to its most common original spelling
tmp = pd.DataFrame({"key": key, "orig": s1}).dropna()
canon = tmp.groupby("key")["orig"].agg(lambda x: x.value_counts().index[0])
df["publisher_clean"] = key.map(canon)

print("AFTER: unique labels =", df["publisher_clean"].nunique())
print(df["publisher_clean"].value_counts().head(10))

# examples of labels that were collapsed together
variants = tmp.groupby("key")["orig"].agg(lambda x: sorted(x.unique()))
variants[variants.str.len() > 1].head(8)

AFTER: unique labels = 13743
publisher_clean
Big Fish Games         286
SEGA                   141
Strategy First         129
Ubisoft                116
Square Enix             95
Sekai Project           94
Choice of Games         92
Dagestan Technology     89
1C Entertainment        85
THQ Nordic              84
Name: count, dtype: int64


key
4 fun studio                              [4 Fun Studio, 4 Fun Studio Inc.]
8 bit development               [8-Bit Development, 8-Bit Development Inc.]
8floor                                                     [8Floor, 8floor]
aegon games                                  [Aegon Games, Aegon Games Ltd]
aerolab studios                      [AeroLab Studios, Aerolab Studios LTD]
aleksandar kuzmanovic games    [Aleksandar Kuzmanovic Games, Aleksandar ...
alexeibelih                                      [Alexeibelih, alexeibelih]
alternative software           [Alternative Software, Alternative Softwa...
Name: orig, dtype: object

## Q5. Extract numbers from text with a regular expression

In [9]:
# owners looks like "10,000,000 .. 20,000,000" (the dots are separated by non-breaking spaces)
ex = df["owners"].astype(str).str.extract(r"^\s*([\d,]+)\D+([\d,]+)\s*$")
df["owners_low"]  = pd.to_numeric(ex[0].str.replace(",", "", regex=False), errors="coerce")
df["owners_high"] = pd.to_numeric(ex[1].str.replace(",", "", regex=False), errors="coerce")
print("owners rows that did NOT match:", df["owners_low"].isna().sum())

# a second example: a trailing number in the game title (Half-Life 2 -> 2)
df["title_number"] = pd.to_numeric(df["game"].str.extract(r"(\d+)\s*$")[0], errors="coerce")
print("title rows that did NOT match :", df["title_number"].isna().sum(), "of", len(df))

df[["owners", "owners_low", "owners_high", "game", "title_number"]].head(8)

owners rows that did NOT match: 0
title rows that did NOT match : 25154 of 26647


,owners,owners_low,owners_high,game,title_number
0,"10,000,000 .. 20,000,000",10000000,20000000,Half-Life 2,2.0
1,"10,000,000 .. 20,000,000",10000000,20000000,Counter-Strike: Source,NaN
2,"10,000,000 .. 20,000,000",10000000,20000000,Counter-Strike: Condition Zero,NaN
3,"5,000,000 .. 10,000,000",5000000,10000000,Half-Life 2: Deathmatch,NaN
4,"2,000,000 .. 5,000,000",2000000,5000000,Half-Life: Source,NaN
5,"1,000,000 .. 2,000,000",1000000,2000000,CS2D,NaN
6,"500,000 .. 1,000,000",500000,1000000,Unreal Tournament 2004: Editor's Choice E...,NaN
7,"500,000 .. 1,000,000",500000,1000000,DOOM 3,3.0


In [10]:
# put owners in its natural order (low -> high) so tables and pivots read correctly
order = df.drop_duplicates("owners_low").sort_values("owners_low")["owners"].astype(str).tolist()
df["owners"] = pd.Categorical(df["owners"].astype(str), categories=order, ordered=True)
print(df["owners"].cat.categories.tolist())

['0\xa0..\xa020,000', '20,000\xa0..\xa050,000', '50,000\xa0..\xa0100,000', '100,000\xa0..\xa0200,000', '200,000\xa0..\xa0500,000', '500,000\xa0..\xa01,000,000', '1,000,000\xa0..\xa02,000,000', '2,000,000\xa0..\xa05,000,000', '5,000,000\xa0..\xa010,000,000', '10,000,000\xa0..\xa020,000,000', '20,000,000\xa0..\xa050,000,000', '50,000,000\xa0..\xa0100,000,000', '100,000,000\xa0..\xa0200,000,000']


## Q6. Parse dates with `errors="coerce"`

In [11]:
raw = df["release_date"].astype(str)
parsed = pd.to_datetime(raw, format="%b %d, %Y", errors="coerce")
n_failed = parsed.isna().sum()
print("Dates that failed the strict 'Mon DD, YYYY' format:", n_failed)
print("Examples:", raw[parsed.isna()].unique()[:6])

# rescue the other formats found in the data, one fallback format at a time
for fmt in ["%b %Y", "%d %b, %Y"]:           # 'Jun 2009' (no day) and '8 Oct, 2014' (day first)
    still = parsed.isna()
    fixed = pd.to_datetime(raw[still], format=fmt, errors="coerce")
    parsed = parsed.fillna(fixed)
    print(f"Rescued with format {fmt!r}: {fixed.notna().sum()}")
print("Still NaT after rescue:", parsed.isna().sum())

# flag the month-only dates, since their day (the 1st) is a guess
df["release_date_approx"] = raw.str.match(r"^[A-Za-z]{3} \d{4}$")

df["release_date"] = parsed
df["release_year"]    = df["release_date"].dt.year.astype("Int16")
df["release_month"]   = df["release_date"].dt.month.astype("Int8")
df["release_dow"]     = df["release_date"].dt.day_name()
df["release_quarter"] = df["release_date"].dt.quarter.astype("Int8")
print(df["release_date"].dtype)
df[["release_date", "release_year", "release_month", "release_dow", "release_quarter"]].head()

Dates that failed the strict 'Mon DD, YYYY' format: 42
Examples: ['Jun 2009' 'Sep 2009' 'Oct 2009' 'Jul 2010' 'Oct 2010' 'Mar 2010']
Rescued with format '%b %Y': 41
Rescued with format '%d %b, %Y': 1
Still NaT after rescue: 0
datetime64[ns]


,release_date,release_year,release_month,release_dow,release_quarter
0,2004-11-16,2004,11,Tuesday,4
1,2004-11-01,2004,11,Monday,4
2,2004-03-01,2004,3,Monday,1
3,2004-11-01,2004,11,Monday,4
4,2004-06-01,2004,6,Tuesday,2


## Q7. `groupby` with named aggregation (5+ aggregations, 1 count, 1 custom function)

In [12]:
def pct_missing(s):
    """Custom aggregation: percentage of missing values in a group."""
    return s.isna().mean() * 100

summary = (df.groupby("release_year")
             .agg(games=("game", "count"),
                  avg_price=("price", "mean"),
                  median_price=("price", "median"),
                  max_avg_playtime=("average_playtime", "max"),
                  publishers=("publisher_clean", "nunique"),
                  pct_price_missing=("price", pct_missing))
             .round(2))
summary

,games,avg_price,median_price,max_avg_playtime,publishers,pct_price_missing
release_year,,,,,,
2004,53,7.71,6.99,371.0,32,5.66
2005,58,8.86,9.99,373.0,42,3.45
2006,112,9.60,9.99,626.0,56,5.36
2007,143,9.45,6.99,1463.0,65,9.09
2008,164,10.58,9.99,1872.0,74,12.80
2009,326,9.59,9.99,419.0,129,5.21
2010,339,9.48,6.99,4901.0,158,7.67
2011,363,9.78,9.99,2156.0,215,6.06
2012,426,12.10,9.99,4106.0,280,9.39


## Q8. Value minus its group mean using `transform`

In [13]:
df["price_minus_year_mean"] = df["price"] - df.groupby("release_year")["price"].transform("mean")
df[["game", "release_year", "price", "price_minus_year_mean"]].head(8)

,game,release_year,price,price_minus_year_mean
0,Half-Life 2,2004,9.99,2.28
1,Counter-Strike: Source,2004,9.99,2.28
2,Counter-Strike: Condition Zero,2004,9.99,2.28
3,Half-Life 2: Deathmatch,2004,4.99,-2.72
4,Half-Life: Source,2004,9.99,2.28
5,CS2D,2004,NaN,NaN
6,Unreal Tournament 2004: Editor's Choice E...,2004,14.99,7.28
7,DOOM 3,2004,4.99,-2.72


`transform("mean")` returns a result with the same length and the same index as the original DataFrame, so it lines up row by row with `price` and can be subtracted directly.
`merge` would need an extra summary table and a join key, which is slower, more code, and can reorder or duplicate rows.

## Q9. Pivot table, then melt back and prove it is lossless

In [14]:
df["price_band"] = pd.cut(df["price"], bins=[-0.01, 2, 10, 30, 1000],
                          labels=["Under $2", "$2-10", "$10-30", "Over $30"])
df["price_band"] = df["price_band"].cat.add_categories("No price listed").fillna("No price listed")

pv = df.pivot_table(index="owners", columns="price_band", values="average_playtime",
                    aggfunc="mean", observed=True)
print(pv.round(1))

price_band                   Under $2       $2-10      $10-30    Over $30  No price listed
owners                                                                                    
0 .. 20,000                  0.200000    0.200000    0.600000    9.200000         0.200000
20,000 .. 50,000             2.200000    1.500000    7.200000   22.400000         3.200000
50,000 .. 100,000            8.300000    1.100000    9.200000    5.100000         6.800000
100,000 .. 200,000          15.200000    4.600000   14.800000   26.500000        11.000000
200,000 .. 500,000          14.700000   17.500000   41.599998  223.500000        11.100000
500,000 .. 1,000,000        20.600000   23.500000   99.300003  275.399994        14.900000
1,000,000 .. 2,000,000      33.799999  195.600006   90.300003  230.600006        51.900002
2,000,000 .. 5,000,000      13.000000   65.300003  266.399994  301.200012       142.100006
5,000,000 .. 10,000,000           NaN  484.899994  781.400024  651.000000       508.200012

In [15]:
long = (pv.reset_index()
          .melt(id_vars="owners", var_name="price_band", value_name="average_playtime")
          .dropna(subset=["average_playtime"]))

ref = (df.groupby(["owners", "price_band"], observed=True)["average_playtime"]
         .mean().reset_index().dropna(subset=["average_playtime"]))

def norm(x):
    x = x.copy()
    x["owners"] = x["owners"].astype(str)
    x["price_band"] = x["price_band"].astype(str)
    return x.sort_values(["owners", "price_band"]).reset_index(drop=True)

pd.testing.assert_frame_equal(norm(long), norm(ref), check_dtype=False)
print("Round trip is lossless: melted table == groupby result,", len(long), "(owners, price_band) cells")

Round trip is lossless: melted table == groupby result, 52 (owners, price_band) cells


## Q10. Split, merge (inner / left / outer) and `validate`

In [16]:
left  = df[["row_id", "game", "price"]].sample(frac=0.8, random_state=1)
right = df[["row_id", "developer", "metascore"]].sample(frac=0.8, random_state=2)
print("left rows :", len(left), "| right rows:", len(right), "| total rows in df:", len(df))

for how in ["inner", "left", "outer"]:
    m = left.merge(right, on="row_id", how=how, validate="one_to_one")
    print(f"{how:>5} merge -> {len(m)} rows")

left rows : 21318 | right rows: 21318 | total rows in df: 26647
inner merge -> 17071 rows
 left merge -> 21318 rows
outer merge -> 25565 rows


- **inner**: only `row_id`s present in BOTH frames (the overlap of the two random 80% samples).
- **left**: every row of `left`, with NaN where `right` has no match (so the count equals `len(left)`).
- **outer**: every `row_id` from either frame (union), NaN on whichever side is missing.

In [17]:
# many-to-many: share a key that repeats on BOTH sides (owners)
small_l = df[["owners", "game"]].head(300)
small_r = df[["owners", "developer"]].head(300)

for rule in ["one_to_one", "one_to_many", "many_to_one", "many_to_many"]:
    try:
        m = small_l.merge(small_r, on="owners", validate=rule)
        print(f"validate='{rule}': OK -> {len(m)} rows (row count blew up from 300 each)")
    except pd.errors.MergeError as e:
        print(f"validate='{rule}': MergeError -> {str(e)[:90]}")

validate='one_to_one': MergeError -> Merge keys are not unique in either left or right dataset; not a one-to-one merge
validate='one_to_many': MergeError -> Merge keys are not unique in left dataset; not a one-to-many merge
validate='many_to_one': MergeError -> Merge keys are not unique in right dataset; not a many-to-one merge
validate='many_to_many': OK -> 12152 rows (row count blew up from 300 each)


## Q11. Rolling mean and rank within each group

In [18]:
df = df.sort_values(["owners", "release_date"]).reset_index(drop=True)

df["price_roll7"] = (df.groupby("owners", observed=True)["price"]
                       .transform(lambda s: s.rolling(7, min_periods=1).mean()))
df["price_rank_in_owners"] = (df.groupby("owners", observed=True)["price"]
                                .rank(method="dense", ascending=False))

df[["game", "owners", "release_date", "price", "price_roll7", "price_rank_in_owners"]].head(10)

,game,owners,release_date,price,price_roll7,price_rank_in_owners
0,KóterGame,"0 .. 20,000",2004-01-11,NaN,NaN,NaN
1,Echelon: Wind Warriors,"0 .. 20,000",2004-02-03,3.99,3.990000,107.0
2,Fair Strike,"0 .. 20,000",2004-02-05,3.99,3.990000,107.0
3,Nemesis of the Roman Empire,"0 .. 20,000",2004-03-17,9.99,5.990000,64.0
4,Lords of the Realm III,"0 .. 20,000",2004-03-17,4.99,5.740000,98.0
5,The Dark Legions,"0 .. 20,000",2004-04-01,1.99,4.990000,135.0
6,Casino Poker,"0 .. 20,000",2004-04-02,2.99,4.656667,121.0
7,Nancy Drew: The Secret of Shadow Ranch,"0 .. 20,000",2004-06-01,9.99,5.418571,64.0
8,Zoo Empire,"0 .. 20,000",2004-08-01,6.99,5.847143,82.0
9,Marine Park Empire,"0 .. 20,000",2004-08-01,6.99,6.275714,82.0


## Q12. The 3 most suspicious rows

In [19]:
r1 = df["price"].idxmax()
r2 = df[df["game"].isna() & df["metascore"].notna()].index[0]
r3 = df[df["game"] == "Half-Life 2: Lost Coast"].index[0]
cols = ["game", "release_date", "price", "owners", "developer", "average_playtime", "median_playtime", "metascore"]
df.loc[[r1, r2, r3], cols]

,game,release_date,price,owners,developer,average_playtime,median_playtime,metascore
12234,ADR-Labelling Game,2018-04-12,595.98999,"0 .. 20,000",Suomen Kuljetusturva Oy,0.0,0.0,NaN
5288,NaN,2017-02-17,8.99000,"0 .. 20,000",Jeroen Wimmers,0.0,0.0,76.0
26613,Half-Life 2: Lost Coast,2005-10-27,NaN,"10,000,000 .. 20,000,000",Valve,0.0,0.0,NaN


1. **ADR-Labelling Game — price 595.99:** the most expensive "game" by far, from a Finnish transport-safety company. It is professional training software, not a game, and the price is an outlier that would distort any price average.
2. **Row with no `game` title but metascore 76:** a game that was reviewed by critics has no name, so the name was lost in scraping and the row cannot be identified.
3. **Half-Life 2: Lost Coast — 10–20 million owners but average playtime 0:** millions of owners and zero minutes played is not believable, so the playtime is most likely missing data stored as 0 (96% of all rows show 0 average playtime).

## Q13. Save as CSV and Parquet, compare size and load time

In [20]:
df.to_csv("video_games_clean.csv", index=False)
df.to_parquet("video_games_clean.parquet", index=False)      # needs pyarrow (already installed in Colab)

csv_kb = os.path.getsize("video_games_clean.csv") / 1024
pq_kb  = os.path.getsize("video_games_clean.parquet") / 1024

def best_time(fn, n=3):
    times = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(); times.append(time.perf_counter() - t0)
    return min(times)

csv_t = best_time(lambda: pd.read_csv("video_games_clean.csv"))
pq_t  = best_time(lambda: pd.read_parquet("video_games_clean.parquet"))

print(f"CSV     : {csv_kb:10.1f} KB | load {csv_t*1000:7.1f} ms")
print(f"Parquet : {pq_kb:10.1f} KB | load {pq_t*1000:7.1f} ms")
print(f"Parquet is {csv_kb/pq_kb:.1f}x smaller and loads {csv_t/pq_t:.1f}x faster")

CSV     :     5079.9 KB | load    99.9 ms
Parquet :     1677.7 KB | load    30.3 ms
Parquet is 3.0x smaller and loads 3.3x faster


## Summary numbers (for the README)

In [21]:
print(f"Dataset rows x cols (after cleaning): {df.shape}")
print(f"Memory saving (Q1)  : {saving_pct:.1f} %  ({mem_before/1024**2:.2f} MB -> {mem_after/1024**2:.2f} MB)")
print(f"CSV size            : {csv_kb:.1f} KB")
print(f"Parquet size        : {pq_kb:.1f} KB")

Dataset rows x cols (after cleaning): (26647, 24)
Memory saving (Q1)  : 26.4 %  (9.56 MB -> 7.03 MB)
CSV size            : 5079.9 KB
Parquet size        : 1677.7 KB
